# 🎯 Golden Dataset 생성 - AI 에이전트 평가를 위한 테스트 데이터셋 만들기

## 📌 이 노트북의 목적

AI 에이전트 평가 시스템을 구축할 때 가장 중요한 것 중 하나는 **Golden Dataset(정답 데이터셋)** 입니다.  
Golden Dataset은 에이전트의 응답을 평가할 때 기준이 되는 **ground truth(정답)** 역할을 합니다.

## 📚 전체 워크플로우

```
PDF 문서들 → pyzerox로 마크다운 변환 → 텍스트 파일로 변환 → LangChain으로 로드 → RAGAS로 테스트셋 생성
```

### 주요 단계:
1. **PDF 파싱**: `pyzerox` 라이브러리를 사용하여 PDF를 마크다운(.md)으로 변환
2. **텍스트 변환**: 마크다운 파일을 텍스트(.txt) 파일로 변환
3. **문서 로딩**: LangChain의 `TextLoader`를 사용하여 문서 로드
4. **테스트셋 생성**: RAGAS의 `TestsetGenerator`를 사용하여 질문-답변 쌍 생성

## 🛠️ 사용하는 주요 라이브러리
- **pyzerox**: PDF를 마크다운으로 변환 (OCR 기반)
- **LangChain**: 문서 로딩 및 처리
- **RAGAS**: 테스트 데이터셋 자동 생성
- **LiteLLM**: 다양한 LLM 제공자 통합 인터페이스

---

**⚠️ 참고**: 아래 에러 메시지는 poppler가 설치되지 않았을 때 발생합니다. Homebrew로 설치하세요: `brew install poppler`

ERROR:root:Error converting PDF to images: Unable to get page count. Is poppler installed and in PATH?

Gemini: https://gemini.google.com/share/2aa89628bab4

## 1️⃣ 환경 설정

`nest_asyncio`는 Jupyter 노트북 환경에서 이미 실행 중인 이벤트 루프 안에서 `asyncio.run()`을 사용할 수 있게 해줍니다.  
`.env` 파일에서 API 키(OpenAI, Anthropic 등)를 로드합니다.

In [ ]:
import nest_asyncio  # Jupyter의 기존 이벤트 루프와 asyncio 호환을 위해 필요
from dotenv import load_dotenv  # .env 파일에서 환경변수(API 키 등)를 로드

nest_asyncio.apply()  # 이벤트 루프 패치 적용
load_dotenv()  # .env 파일의 환경변수를 os.environ에 등록

## 2️⃣ PDF를 마크다운으로 변환 (pyzerox)

`pyzerox`는 OCR 기반 PDF 파싱 라이브러리입니다. GPT-4.1 모델을 활용하여 PDF의 텍스트와 구조를 인식한 뒤 마크다운 형식으로 변환합니다.

- 입력: `documents_with_english_titles/` 폴더의 PDF 파일들
- 출력: `documents_with_english_titles_markdown/` 폴더에 `.md` 파일 생성

In [ ]:
from pyzerox import zerox  # OCR 기반 PDF → 마크다운 변환 라이브러리
import os
import asyncio

model = 'gpt-4.1'  # pyzerox가 PDF를 해석할 때 사용할 LLM 모델

async def main():
    """
    documents_with_english_titles 폴더의 모든 PDF를 마크다운으로 변환합니다. 
    """

    # pyzerox 옵션 설정
    kwargs = {}
    custom_system_prompt = None  # 기본 시스템 프롬프트 사용
    select_pages = None  # None이면 모든 페이지를 변환
    output_dir = 'documents_with_english_titles_markdown'  # 변환 결과 저장 디렉토리

    # 폴더 내 모든 PDF 파일을 순회하며 변환
    for file_path in os.listdir("./documents_with_english_titles"):
        file_path = os.path.join("./documents_with_english_titles", file_path)
        # PDF를 마크다운으로 변환 (비동기 함수 - LLM API 호출 포함)
        result = await zerox(
            file_path=file_path,
            model=model,
            output_dir=output_dir,
            custom_system_prompt=custom_system_prompt,
            select_pages=select_pages,
            **kwargs
        )
        print(f"Converted {file_path} to markdown.")
        print(f'result == {result}')

# asyncio.run()으로 비동기 함수 실행 (nest_asyncio 덕분에 Jupyter에서도 동작)
result = asyncio.run(main())

## 3️⃣ 마크다운을 텍스트 파일로 변환

RAGAS의 `TestsetGenerator`가 요구하는 입력 형식에 맞추기 위해 `.md` 파일을 `.txt` 파일로 변환합니다.  
내용은 동일하며, 확장자만 변경됩니다.

In [ ]:
from pathlib import Path

input_dir = Path('./documents_with_english_titles_markdown')  # 마크다운 파일이 있는 디렉토리
output_dir = Path('./documents_with_english_titles_txt')  # 텍스트 파일을 저장할 디렉토리

# 출력 디렉토리가 없으면 생성
output_dir.mkdir(exist_ok=True)

# 모든 마크다운 파일을 텍스트 파일로 변환
for md_file in input_dir.glob("*.md"):
    # 마크다운 파일 내용 읽기 (UTF-8 인코딩)
    text_content = md_file.read_text(encoding='utf-8')
    
    # 확장자만 .txt로 변경하여 저장 (내용은 동일)
    txt_file = output_dir / md_file.with_suffix('.txt').name
    txt_file.write_text(text_content, encoding='utf-8')

## 4️⃣ 문서 로딩 (LangChain)

LangChain의 `DirectoryLoader`를 사용하여 텍스트 파일들을 `Document` 객체로 로드합니다.  
각 `Document`에는 파일 내용(`page_content`)과 메타데이터(`metadata.source`)가 포함됩니다.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# DirectoryLoader: 디렉토리 내 파일들을 일괄 로드하는 LangChain 유틸리티
# glob 패턴으로 .txt 파일만 선택
loader = DirectoryLoader('./documents_with_english_titles_txt', glob='**/*.txt')
documents = loader.load()  # List[Document] 반환

In [ ]:
# 로드된 문서 중 하나를 확인 - metadata.source에 원본 파일 경로가 기록됨
documents[4]

## 5️⃣ LLM 및 임베딩 모델 설정

RAGAS의 `TestsetGenerator`에는 두 가지 모델이 필요합니다:

1. **Generator LLM**: 질문-답변 쌍을 생성하는 언어 모델 (여기서는 Claude 사용)
2. **Embedding 모델**: 문서 간 의미적 유사도를 계산하는 임베딩 모델 (여기서는 OpenAI 사용)

`LiteLLM`을 사용하면 다양한 LLM 제공자(OpenAI, Anthropic 등)를 통합된 인터페이스로 호출할 수 있습니다.

In [ ]:
# === LLM 및 임베딩 모델 설정 ===
# RAGAS 테스트셋 생성에 필요한 모델들을 초기화합니다

import litellm  # 다양한 LLM 제공자를 통합하는 라이브러리
import openai
from ragas.llms import llm_factory  # RAGAS용 LLM 래퍼 생성
from ragas.embeddings import OpenAIEmbeddings  # 임베딩 모델 래퍼

# ===== LLM 설정 (Claude 사용) =====
# LiteLLM을 통해 Anthropic의 Claude 모델을 사용합니다
# llm_factory는 RAGAS가 사용할 수 있는 형태로 LLM을 래핑합니다

generator_llm = llm_factory(
    "anthropic/claude-sonnet-4-5",  # LiteLLM 형식: "제공자/모델명"
    provider="litellm",  # LiteLLM 프로바이더 사용
    client=litellm.completion,  # LiteLLM의 completion 함수 전달
    temperature=0.1,  # 낮은 temperature = 더 일관된 출력
    top_p=None,  # temperature와 충돌 방지를 위해 비활성화
    max_tokens=64000,  # 최대 출력 토큰 수
)

# ===== 임베딩 모델 설정 (OpenAI 사용) =====
# 문서 간 의미적 유사도 계산을 위한 임베딩 모델
# text-embedding-3-large는 OpenAI의 최신 고성능 임베딩 모델입니다

openai_client = openai.OpenAI()  # OpenAI 클라이언트 초기화
generator_embeddings = OpenAIEmbeddings(
    client=openai_client,
    model="text-embedding-3-large"  # 고품질 임베딩
)

## 6️⃣ 생성된 테스트셋 확인

RAGAS가 생성한 테스트셋은 CSV 파일로 저장되어 있습니다.  
각 행에는 `user_input`(질문), `reference`(정답), `reference_contexts`(참조 문서), `synthesizer_name`(생성 방식) 등의 컬럼이 포함되어 있습니다.

RAGAS는 다양한 유형의 질문을 자동 생성합니다:
- **single_hop**: 하나의 문서에서 답을 찾을 수 있는 질문
- **multi_hop**: 여러 문서를 조합해야 답할 수 있는 질문
- 다양한 `query_style` (완벽한 문법, 맞춤법 오류, 웹 검색 스타일 등)으로 현실적인 질문 패턴을 모사합니다.

In [ ]:
import pandas as pd

# RAGAS가 생성한 테스트셋 CSV 로드
df_ragas = pd.read_csv('./ragas_testset.csv')

In [ ]:
# 전체 데이터프레임 출력 - 생성된 질문-답변 쌍과 메타데이터 확인
df_ragas